# Day 8 — U-Net and Skip Connections (LandCover.ai)

First semantic segmentation notebook: dense per-pixel prediction on aerial imagery,
5 classes, 512×512 tiles downsampled to 256×256.

The goal is **not** a good IoU. It is to establish two things with measured evidence:
what skip connections actually contribute, and why pixel accuracy is the wrong metric
for imbalanced segmentation.

## Result

| metric | U-Net (skip) | U-Net (no skip) | change |
|---|---|---|---|
| pixel accuracy | 0.8992 | 0.8933 | **−0.6%** |
| mIoU | 0.5626 | 0.4643 | −17.5% |
| background (29.6% of pixels) | 0.856 | 0.845 | −1.3% |
| woodland (60.9%) | 0.828 | 0.822 | −0.7% |
| water (8.5%) | 0.606 | 0.655 | +8.1% |
| **building (0.23%)** | **0.353** | **0.000** | **−100%** |
| **road (0.81%)** | **0.170** | **0.000** | **−100%** |

**Pixel accuracy differs by 0.6 points. Behind that number, two entire classes
disappeared** — the no-skip model never predicted a single building or road pixel in
six epochs.

Judged on accuracy alone, skip connections would look irrelevant. That is the whole
lesson.

## Takeaways

1. **Pixel accuracy lies under class imbalance.** Predicting nothing but woodland
   scores 60.9% before learning anything.
2. **Skip connections supply spatial localization, not capacity.** Large regions lose
   ~1%; small and thin objects go to zero. That selectivity rules out the 11%
   parameter difference as an explanation.
3. **Plain cross-entropy abandons rare classes.** After epoch 1: building 0.000,
   water 0.010, road 0.000 — while pixel accuracy was already 0.80.

## 1. Data — LandCover.ai

Aerial orthophotos from Poland at 25–50 cm resolution, manually annotated. Used here
via a pre-tiled redistribution (512×512 chips, official train/val/test split already
applied).

**License: CC-BY-NC-SA-4.0** — non-commercial, share-alike. Original dataset:
<https://landcover.ai.linuxpolska.com/>. Cite Boguszewski et al., CVPRW 2021
(doi:10.1109/CVPRW53098.2021.00121).

The official splits are cut by **orthophoto**, not by random chip — different splits
come from different aerial images. That sidesteps the spatial-autocorrelation leakage
that random chip splitting would introduce, which is the subject of Day 11.

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os, glob

zp = hf_hub_download(
    repo_id="MortenTabaka/LandCover-Aerial-Imagery-for-semantic-segmentation",
    filename="landcover_processed_for_training.zip",
    repo_type="dataset",
)
DEST = "/content/landcover"
if not os.path.exists(f"{DEST}/processed/train"):
    with zipfile.ZipFile(zp) as z:
        z.extractall(DEST)

for split in ("train", "val", "test"):
    n_i = len(glob.glob(f"{DEST}/processed/{split}/images/**/*.jpg", recursive=True))
    n_m = len(glob.glob(f"{DEST}/processed/{split}/masks/**/*.png",  recursive=True))
    print(f"{split:6s} images {n_i:6d}  masks {n_m:6d}")

### Inspecting before assuming

The HuggingFace `datasets` loader flattened this archive into a single `image` column
with no masks — 21,348 rows, which is 2 × 10,674, i.e. images and masks concatenated
with the directory structure discarded. Reading the zip directly recovers the pairing.

Worth doing in general: **print the structure rather than assume it**. Three things
needed checking here, and two of them would have caused silent failures.

In [ ]:
import numpy as np
from PIL import Image
from collections import Counter

train_masks = sorted(glob.glob(f"{DEST}/processed/train/masks/**/*.png", recursive=True))
train_imgs  = sorted(glob.glob(f"{DEST}/processed/train/images/**/*.jpg", recursive=True))

m = np.array(Image.open(train_masks[0]))
print("mask shape:", m.shape, m.dtype)
print("channels identical:",
      np.array_equal(m[:,:,0], m[:,:,1]) and np.array_equal(m[:,:,1], m[:,:,2]))

cnt = Counter()
for p in train_masks[:300]:
    a = np.array(Image.open(p))
    a = a[:, :, 0] if a.ndim == 3 else a
    u, c = np.unique(a, return_counts=True)
    cnt.update(dict(zip(u.tolist(), c.tolist())))

total = sum(cnt.values())
print(f"\nclass distribution over 300 tiles ({total/1e6:.1f}M pixels):")
for k in sorted(cnt):
    print(f"  value {k}: {cnt[k]/total*100:6.2f}%")

def stem(p):
    return os.path.splitext(os.path.basename(p))[0]
print("\nfilename stems match:", set(map(stem, train_imgs)) == set(map(stem, train_masks)))

### The number that drives everything else

| value | class | share of pixels |
|---|---|---|
| 0 | background | 29.61% |
| **1** | **building** | **0.23%** |
| 2 | woodland | 60.89% |
| 3 | water | 8.45% |
| 4 | road | 0.81% |

**Woodland outnumbers building 265 to 1.**

A model that predicts nothing but woodland gets 60.9% pixel accuracy. Add background
and water and it clears 90% — with a building IoU of zero.

For comparison, CIFAR-10 in Day 6 was perfectly balanced at 10% per class, and per-class
F1 still spread 20 points. Here the smallest class is 0.23%.

This notebook deliberately uses plain `CrossEntropyLoss` so the resulting failure is
visible rather than described. Day 9 fixes it.

## 2. Dataset

Two details here cause **silent** failures rather than errors:

**Masks must be resized with `NEAREST`.** Bilinear interpolation on class indices
invents values that do not exist — at a boundary between building (1) and water (3) it
produces 2, i.e. woodland pixels that were never labelled. No error, corrupted labels.

**Image/mask pairing must be verified explicitly.** If the two `glob` calls return
different orderings, the model trains image A against mask B. Loss still decreases
(the model learns the class prior), IoU is inexplicably low, and nothing raises an
exception. The assertion below is the only defence.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

SIZE = 256          # downsampled from 512 for iteration speed
N_CLASSES = 5
CLASS_NAMES = ["background", "building", "woodland", "water", "road"]

# ImageNet statistics, not this dataset's -- chosen for forward compatibility
# with the pretrained encoders introduced on Day 12.
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def _stem(p):
    return os.path.splitext(os.path.basename(p))[0]

class LandCoverDS(Dataset):
    def __init__(self, split, size=SIZE, limit=None):
        root = f"{DEST}/processed/{split}"
        self.imgs = sorted(glob.glob(f"{root}/images/**/*.jpg", recursive=True))
        self.msks = sorted(glob.glob(f"{root}/masks/**/*.png",  recursive=True))
        assert len(self.imgs) == len(self.msks) > 0, f"{split}: {len(self.imgs)}/{len(self.msks)}"
        bad = [(a, b) for a, b in zip(self.imgs, self.msks) if _stem(a) != _stem(b)]
        assert not bad, f"{len(bad)} mismatched pairs, first: {bad[0]}"
        if limit:
            self.imgs, self.msks = self.imgs[:limit], self.msks[:limit]
        self.size = size

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        img = Image.open(self.imgs[i]).convert("RGB").resize((self.size, self.size), Image.BILINEAR)
        msk = Image.open(self.msks[i]).resize((self.size, self.size), Image.NEAREST)

        x = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0
        x = (x - MEAN) / STD

        m = np.array(msk)
        m = m[:, :, 0] if m.ndim == 3 else m      # three identical channels -> take one
        return x, torch.from_numpy(m.astype(np.int64))

train_ds = LandCoverDS("train")
val_ds   = LandCoverDS("val")
test_ds  = LandCoverDS("test")
print(len(train_ds), len(val_ds), len(test_ds))

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

xb, yb = next(iter(train_dl))
print("image:", xb.shape, xb.dtype, f"[{xb.min():.2f}, {xb.max():.2f}]")
print("mask :", yb.shape, yb.dtype, "unique:", torch.unique(yb).tolist())

### Shape change from classification

| | classification (Days 3–6) | segmentation |
|---|---|---|
| output | `[N, 5]` | `[N, 5, H, W]` |
| target | `[N]` | `[N, H, W]` |
| main metric | accuracy | IoU |

The mask has **no channel dimension** — `nn.CrossEntropyLoss` accepts `[N,C,H,W]`
logits against `[N,H,W]` integer targets directly and averages over all pixels. No
reshaping, no one-hot encoding.

## 3. U-Net

Three parts:

- **Encoder** — ordinary CNN. conv + pool repeatedly: spatial size down, channels up.
  Extracts semantics, discards precise position.
- **Decoder** — transposed convolutions upsample back to the input resolution.
- **Skip connections** — concatenate each encoder level's feature map into the
  matching decoder level.

Why skips are needed: by the bottleneck, 256×256 has been compressed to 16×16. The
model knows *there is a building in this area* but not *which pixels*. Upsampling
cannot recreate discarded position information. Skips carry high-resolution shallow
features across directly.

**Deep layers know what; shallow layers know where; skips join them.**

Note this is `concat`, not the `add` used by ResNet's residual connections
(`F(x) + x`). Concatenation keeps both signals separate and lets the following conv
decide how to weigh them.

`use_skip` is a constructor flag so that the ablation differs by exactly one boolean —
not two separately written models that might diverge in other ways.

In [ ]:
import torch.nn as nn

class DoubleConv(nn.Module):
    """conv-BN-ReLU twice -- the basic block at every U-Net level."""
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=N_CLASSES, base=32, use_skip=True):
        super().__init__()
        self.use_skip = use_skip
        b = base
        self.pool = nn.MaxPool2d(2)

        self.enc1 = DoubleConv(in_ch, b)          # 256
        self.enc2 = DoubleConv(b,   b*2)          # 128
        self.enc3 = DoubleConv(b*2, b*4)          # 64
        self.enc4 = DoubleConv(b*4, b*8)          # 32
        self.bottleneck = DoubleConv(b*8, b*16)   # 16

        # Skip concatenation doubles the decoder's input channels.
        s = (lambda c: c if use_skip else 0)
        self.up4  = nn.ConvTranspose2d(b*16, b*8, 2, stride=2)
        self.dec4 = DoubleConv(b*8 + s(b*8), b*8)
        self.up3  = nn.ConvTranspose2d(b*8, b*4, 2, stride=2)
        self.dec3 = DoubleConv(b*4 + s(b*4), b*4)
        self.up2  = nn.ConvTranspose2d(b*4, b*2, 2, stride=2)
        self.dec2 = DoubleConv(b*2 + s(b*2), b*2)
        self.up1  = nn.ConvTranspose2d(b*2, b, 2, stride=2)
        self.dec1 = DoubleConv(b + s(b), b)

        self.head = nn.Conv2d(b, n_classes, 1)    # 1x1 conv -> per-pixel logits

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        bn = self.bottleneck(self.pool(e4))

        d4 = self.up4(bn); d4 = self.dec4(torch.cat([d4, e4], 1) if self.use_skip else d4)
        d3 = self.up3(d4); d3 = self.dec3(torch.cat([d3, e3], 1) if self.use_skip else d3)
        d2 = self.up2(d3); d2 = self.dec2(torch.cat([d2, e2], 1) if self.use_skip else d2)
        d1 = self.up1(d2); d1 = self.dec1(torch.cat([d1, e1], 1) if self.use_skip else d1)
        return self.head(d1)

device = "cuda" if torch.cuda.is_available() else "cpu"
for flag in (True, False):
    m = UNet(use_skip=flag).to(device)
    out = m(xb[:2].to(device))
    print(f"use_skip={flag!s:5s}  params {sum(p.numel() for p in m.parameters()):,}  "
          f"out {tuple(out.shape)}")

The output head is a **1×1 convolution** mapping `base` channels to 5 class logits.
A 1×1 conv does no spatial mixing — it is a fully connected layer applied
independently at every pixel position. Structurally the same object as Day 3's
`Linear(128, 10)`, evaluated 65,536 times instead of once.

**Note the parameter gap: 7,763,173 vs 6,979,813, an 11.2% difference.** Skip
concatenation doubles the decoder's input channels, so the no-skip model is
necessarily smaller. That is a confounder for the ablation, addressed after the
results.

## 4. Training and IoU

**IoU must be computed from a confusion matrix accumulated over the whole loader**,
not averaged over per-batch IoUs. With building at 0.23% of pixels, many batches
contain no buildings at all; their building IoU is 0 or undefined, and averaging those
in produces a meaningless number.

This ordering error is common in published remote-sensing segmentation work. Accumulate
first, divide once.

`(yb * n_classes + pred)` with `bincount` flattens a 2-D confusion matrix into 1-D
counts — orders of magnitude faster than looping over classes.

In [ ]:
import time, copy

@torch.no_grad()
def evaluate(model, dl, n_classes=N_CLASSES):
    model.eval()
    cm = torch.zeros(n_classes, n_classes, dtype=torch.int64, device=device)
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(1)
        k = (yb * n_classes + pred).flatten()
        cm += torch.bincount(k, minlength=n_classes**2).reshape(n_classes, n_classes)

    cm = cm.float()
    tp = cm.diag()
    union = cm.sum(0) + cm.sum(1) - tp
    iou = tp / union.clamp(min=1)
    present = cm.sum(1) > 0                      # classes present in ground truth
    return dict(pixel_acc=(tp.sum()/cm.sum()).item(),
                miou=iou[present].mean().item(),
                iou=iou.cpu().numpy(), cm=cm.cpu().numpy())

def train_seg(model, epochs=6, lr=1e-3, tag="run"):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()              # plain CE, deliberately
    best = {"miou": -1}; hist = []

    for ep in range(epochs):
        t0 = time.time(); model.train(); running = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)        # [N,5,H,W] vs [N,H,W]
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item() * xb.size(0)

        m = evaluate(model, val_dl); hist.append(m)
        per_cls = "  ".join(f"{n[:5]} {v:.3f}" for n, v in zip(CLASS_NAMES, m["iou"]))
        print(f"[{tag}] ep {ep+1}  loss {running/len(train_ds):.4f}  "
              f"pixAcc {m['pixel_acc']:.4f}  mIoU {m['miou']:.4f}  ({time.time()-t0:.0f}s)")
        print(f"         {per_cls}")

        if m["miou"] > best["miou"]:
            best = m
            best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})

    model.load_state_dict(best_state)
    return model, hist, best

### 4a. Baseline — with skip connections

In [ ]:
torch.manual_seed(42)
unet, hist_unet, best_unet = train_seg(UNet(use_skip=True), epochs=6, tag="unet")

**Epoch 1 is the cleanest evidence of what cross-entropy does here:**

```
pixAcc 0.8028   building 0.000   water 0.010   road 0.000
```

80% pixel accuracy while three of five classes are untouched. Background and woodland
together are 90.5% of pixels, so predicting only those two is already worth 0.80.
Cross-entropy weights every pixel equally, so a class at 0.23% contributes almost
nothing to the loss — **the model is correctly optimizing the wrong objective**.

Final IoU tracks class frequency almost monotonically:

| class | share | IoU |
|---|---|---|
| background | 29.61% | 0.856 |
| woodland | 60.89% | 0.828 |
| water | 8.45% | 0.606 |
| road | 0.81% | **0.170** |
| building | 0.23% | **0.353** |

The exception is informative: **building scores higher than road despite being 3.5×
rarer.** Buildings are compact blocks; roads are thin lines. Thin structures have a far
higher boundary-to-area ratio — a 10-pixel-wide road displaced by 2 pixels loses ~40%
of its intersection, while a 50×50 building displaced by 2 pixels loses ~8%.

So road suffers from two compounding problems (rarity *and* shape); building from one.
Conflating them would lead to the wrong intervention.

mIoU was still rising at epoch 6 (0.555 → 0.563). `epochs=6` is an arbitrary cap, the
same censoring issue found in Days 4 and 6 — acceptable here because the ablation
compares two runs under identical conditions, but it means these numbers are floors.

### 4b. Ablation — same model, no skip connections

In [ ]:
torch.manual_seed(42)
unet_ns, hist_ns, best_ns = train_seg(UNet(use_skip=False), epochs=6, tag="no-skip")

### Reading the ablation

**Building and road are exactly 0.000 in all six epochs.** Not low — the model never
predicted a single pixel of either class.

| metric | skip | no skip | change |
|---|---|---|---|
| pixel accuracy | 0.8992 | 0.8933 | −0.6% |
| mIoU | 0.5626 | 0.4643 | −17.5% |
| background (29.6%) | 0.856 | 0.845 | −1.3% |
| woodland (60.9%) | 0.828 | 0.822 | −0.7% |
| water (8.5%) | 0.606 | 0.655 | +8.1% |
| **building (0.23%)** | **0.353** | **0.000** | **−100%** |
| **road (0.81%)** | **0.170** | **0.000** | **−100%** |

**Pixel accuracy differs by 0.6 points while two classes vanish entirely.** Judged on
accuracy, skip connections look like noise.

**The 11% parameter confounder is ruled out by the selectivity of the failure.** The
no-skip model matches the skip model on woodland (0.822 vs 0.828) and background
(0.845 vs 0.856), so it is not capacity-limited overall. An 11% parameter reduction
cannot produce "two specific classes disappear while the others are unaffected". This
is a qualitative loss of localization, not a quantitative loss of capacity.

**Water going *up* (0.606 → 0.655) is not interpretable from a single run.** That
column is highly unstable — the no-skip run went 0.431 → 0.322 → 0.655 over the last
three epochs. Comparing each run's best gives 0.631 vs 0.655, a 2.4% gap, well inside
the run-to-run variance measured in Day 6. There is also a plausible mechanism (with
building and road never predicted, those pixels are redistributed, and water competes
with fewer classes), but distinguishing the two would need multiple seeds. The
building/road result needs no such caveat: 0.353 → 0.000 is categorical.

### What skip connections do

The encoder compresses 256×256 to 16×16 — a 16× loss of spatial resolution. At the
bottleneck the model knows *a building is somewhere in this region* but not *which
pixels*. Upsampling cannot recover discarded position information; skips carry
high-resolution shallow features across directly.

- **Large contiguous regions** (woodland, background): a few pixels of localization
  error barely changes IoU → ~1% drop
- **Small or thin objects** (building, road): localization error consumes the entire
  object → total loss

## 5. Visual comparison

Random validation tiles almost never contain buildings (0.23% of pixels), so tiles are
selected by building content first.

`interpolation="nearest"` is required — the default blends class indices at boundaries,
making segmentation output look blurrier than it is. Same reasoning as `NEAREST` for
mask resizing.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# background, building, woodland, water, road
PALETTE = ListedColormap(["#4d4d4d", "#e41a1c", "#4daf4a", "#377eb8", "#ff7f00"])

@torch.no_grad()
def predict_batch(model, xb):
    model.eval()
    return model(xb.to(device)).argmax(1).cpu()

picks = []
for i in range(len(val_ds)):
    _, m = val_ds[i]
    if (m == 1).float().mean() > 0.03:      # >3% building pixels
        picks.append(i)
    if len(picks) == 4:
        break
print("tiles with buildings:", picks)

xs = torch.stack([val_ds[i][0] for i in picks])
ys = torch.stack([val_ds[i][1] for i in picks])
p_skip = predict_batch(unet,    xs)
p_none = predict_batch(unet_ns, xs)

mu = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
sd = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

rows = ["RGB", "ground truth", "U-Net (skip)", "no skip"]
fig, axes = plt.subplots(4, len(picks), figsize=(3.2*len(picks), 13))
for j in range(len(picks)):
    axes[0][j].imshow((xs[j]*sd+mu).permute(1,2,0).clamp(0,1).numpy())
    axes[1][j].imshow(ys[j],     cmap=PALETTE, vmin=0, vmax=4, interpolation="nearest")
    axes[2][j].imshow(p_skip[j], cmap=PALETTE, vmin=0, vmax=4, interpolation="nearest")
    axes[3][j].imshow(p_none[j], cmap=PALETTE, vmin=0, vmax=4, interpolation="nearest")
    for r in range(4):
        axes[r][j].axis("off")
        if j == 0:
            axes[r][j].set_title(rows[r], loc="left", fontsize=10)
plt.suptitle("Skip connections: red = building, orange = road", fontsize=11)
plt.tight_layout(); plt.show()

### What the figure shows

**Row 4 (no skip): not a single red or orange pixel.** Only grey and green. Against
row 2, where regular red building blocks and an orange road network run across the
frame. Same input, both classes erased into background. This is what IoU 0.000 looks
like — not inaccurate, absent.

**Row 3 (skip): present but crude.** Red appears roughly where buildings are, and the
road is visible as a broken line. But the shapes are wrong: ground-truth buildings are
clean rectangles, predictions are fragmented blobs with ragged edges.

That is exactly what a building IoU of 0.353 looks like: **position right, shape
wrong.** IoU penalizes the intersection-over-union, so a correctly located but
fragmented prediction scores poorly.

So the conclusion has three levels, not two:

| | localization | outcome |
|---|---|---|
| no skip | none | small objects vanish (IoU 0.000) |
| skip, 6 epochs | coarse | right place, wrong shape (IoU 0.353) |
| skip + imbalance handling + longer training | — | Day 9 |

**Skip connections give the model localization; they do not make localization
precise.** Reading "0 → 0.35" as problem-solved would be wrong — the figure shows how
far off it still is.

One reading to avoid: in row 3 the predicted woodland is visibly larger than ground
truth. That is a different kind of error. Woodland and background (grass, farmland)
genuinely grade into each other in aerial RGB, and the annotation boundary is partly
subjective. **Boundary blur between large classes is a localization-precision and
label-ambiguity problem; total disappearance of small classes is a loss-function
problem.** They need different fixes.

## Limitations

- **Six epochs, chosen arbitrarily.** mIoU was still rising for both models when
  training stopped (skip: 0.555 → 0.563). Both numbers are floors. No early stopping
  or patience criterion was used — the Day 6 discipline is deliberately deferred here
  so the ablation stays simple, but it should be applied from Day 9.
- **Single seed.** Differences of 1–2 points (notably the water column) are within
  run-to-run variance and not interpretable; see Day 6 §4. The building/road result is
  categorical and does not depend on this.
- **11% parameter difference between arms** is inherent to the ablation — removing
  skips necessarily shrinks the decoder. Argued to be non-explanatory above, but not
  controlled for directly. A width-matched no-skip model would settle it.
- **Validation used for model selection** (best-mIoU checkpoint), and test is untouched
  so far. No final test evaluation is reported in this notebook.
- **512 → 256 downsampling** discards detail that matters most for the thin and small
  classes, so building and road IoU here are pessimistic relative to full resolution.

## Summary

| | U-Net (skip) | U-Net (no skip) |
|---|---|---|
| pixel accuracy | 0.8992 | 0.8933 |
| mIoU | **0.5626** | 0.4643 |
| building IoU | **0.353** | **0.000** |
| road IoU | **0.170** | **0.000** |

1. **Pixel accuracy is the wrong metric under class imbalance.** A 0.6-point
   difference concealed the complete loss of two classes.
2. **Skip connections supply spatial localization, not capacity.** Large classes lose
   ~1%; small and thin classes go to zero. The selectivity of that failure is what
   rules out the parameter-count confounder.
3. **Plain cross-entropy abandons rare classes**, because a class at 0.23% of pixels
   barely affects a per-pixel mean loss.
4. **Thin structures are harder than rare ones.** Road (0.81%) scores below building
   (0.23%) because thin shapes lose proportionally more to boundary error.

**Next:** loss functions for imbalanced segmentation — Dice, class-weighted
cross-entropy, and combinations — evaluated against per-class IoU rather than accuracy.